# 存储拍摄与 WiFi 原件导出

需要支持 MEDIA_CATALOG 的 EDU BT；ISP/WiFi 保持公版。只导出有真实媒体 ID 的 JPEG/MP4 和已登记伴随文件，默认保留眼镜原件；视频是设备原件，不执行手机 App 防抖。
macOS 单元格在内核主线程运行，等待使用 s.wait()。

In [ ]:
from pathlib import Path
import json
from edu_host import EduSession, WifiSecurity

s = EduSession(bt="auto")
s.connect()
s.media_capabilities()


In [ ]:
photo = s.capture_saved_photo()
photo  # 公版控制状态和 operation_id，不保证本次媒体 ID 或发布完成


In [ ]:
recording = s.video_start()
s.wait(5)
video = s.video_stop(recording.operation_id)
video


将网络凭据保存在本地 wifi-test.json：包含 ssid、security（例如 WPA2）、password；不要将真实凭据提交到仓库或保存到 Notebook 输出。PC 与眼镜需在可达局域网，PC 可以使用有线网络。

In [ ]:
credentials = json.loads(Path("wifi-test.json").read_text())
with s.media_import(credentials["ssid"], credentials.get("password", ""),
                    WifiSecurity[credentials.get("security", "WPA2")]) as media:
    items = media.list()
    print([(item.media_id, item.file.name) for item in items])
    for item in items:  # 选择真实索引条目，不猜测本次拍摄的 ID
        result = media.download(item, "exports")
        print(result.main_success, result.complete, result.files, result.errors)
    # 批量：results = media.download_all("exports")
    # 显式删除（单独决定后执行）：media.delete(items[0])


MP4 成功而 gyro 缺失/下载失败或公版 legacy 目录无法确认关联时，main_success 为 True，complete 为 False，并在 errors 中列出问题。批量下载逐项返回结果，普通文件失败继续其他项；CTRL 或 LAN 会话失效时终止。下载失败整文件有限重试，不支持断点续传。

In [ ]:
s.close()
